# MOD-06 · NB-02 — Propensity Score Methods
### Health Analytics with Python · Module 06: Causal Inference in Health Research
---
**Learning objectives**
- Estimate propensity scores using logistic regression and GBM
- Apply PS matching (nearest-neighbour, caliper, 1:k)
- Implement inverse probability of treatment weighting (IPTW)
- Assess covariate balance with Standardised Mean Differences (SMD)
- Estimate ATE and ATT with confidence intervals via bootstrap
- Compare PS matching vs IPTW trade-offs

**Estimated time:** 3 hours | **Level:** Intermediate-Advanced | **Libraries:** `sklearn`, `scipy`, `pandas`


## 1. Setup & Shared Dataset

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore"); import os; os.makedirs("/tmp/mod06", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
np.random.seed(42); N = 5000
def sigmoid(x): return 1/(1+np.exp(-x))
age = np.random.normal(60,12,N).clip(30,90)
smoking = np.random.binomial(1,sigmoid(-1+0.02*(age-60)),N)
chd_history = np.random.binomial(1,sigmoid(-2+0.03*(age-60)),N)
ldl_baseline = np.random.normal(140+20*smoking,30,N).clip(60,280)
ses_score = np.random.normal(0,1,N)
statin_logit = -2.0+0.025*(ldl_baseline-140)+1.0*chd_history+0.02*(age-60)-0.3*ses_score
statin = np.random.binomial(1,sigmoid(statin_logit),N)
TRUE_EFFECT = -0.8
mi_logit = -3.0+TRUE_EFFECT*statin+0.03*(age-60)+0.5*smoking+0.8*chd_history+0.005*(ldl_baseline-140)
mi = np.random.binomial(1,sigmoid(mi_logit),N)
df = pd.DataFrame({"age":age,"smoking":smoking,"chd_history":chd_history,
                   "ldl_baseline":ldl_baseline,"ses_score":ses_score,"statin":statin,"mi":mi})
COVARIATES = ["age","smoking","chd_history","ldl_baseline","ses_score"]
print(f"N={N} | Statin={statin.mean()*100:.1f}% | MI={mi.mean()*100:.1f}% | True OR={np.exp(TRUE_EFFECT):.3f}")

## 2. Propensity Score Estimation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import calibration_curve
import seaborn as sns

# Estimate PS with logistic regression
lr_ps = LogisticRegression(C=1, max_iter=1000)
lr_ps.fit(df[COVARIATES], df["statin"])
df["ps_lr"] = lr_ps.predict_proba(df[COVARIATES])[:,1]

# Estimate PS with gradient boosting (more flexible)
gb_ps = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                    subsample=0.8, random_state=42)
gb_ps.fit(df[COVARIATES], df["statin"])
df["ps_gb"] = gb_ps.predict_proba(df[COVARIATES])[:,1]

fig, axes = plt.subplots(1,3,figsize=(16,4))
# PS distribution by treatment group
for ps_col, ax, title in [("ps_lr",axes[0],"Logistic Regression PS"),
                           ("ps_gb",axes[1],"Gradient Boosting PS")]:
    for grp, color, label in [(0,"#4878CF","Control (no statin)"),(1,"#D65F5F","Treated (statin)")]:
        sub = df[df.statin==grp][ps_col]
        ax.hist(sub, bins=40, alpha=0.6, color=color, label=label, density=True)
    ax.set_xlabel("Propensity score"); ax.set_ylabel("Density")
    ax.set_title(title); ax.legend(fontsize=8)

# Calibration
fp, mp = calibration_curve(df["statin"], df["ps_gb"], n_bins=10)
axes[2].plot(mp, fp, "o-", color="#D65F5F", label="GBM PS")
fp2, mp2 = calibration_curve(df["statin"], df["ps_lr"], n_bins=10)
axes[2].plot(mp2, fp2, "s--", color="#4878CF", label="LR PS")
axes[2].plot([0,1],[0,1],"k--",lw=1)
axes[2].set_xlabel("Mean predicted PS"); axes[2].set_ylabel("Fraction treated")
axes[2].set_title("PS Calibration"); axes[2].legend(fontsize=8)

plt.tight_layout(); plt.savefig("/tmp/mod06/ps_distributions.png",bbox_inches="tight"); plt.show()
print(f"PS range (LR): {df.ps_lr.min():.3f} - {df.ps_lr.max():.3f}")
print(f"PS range (GB): {df.ps_gb.min():.3f} - {df.ps_gb.max():.3f}")


## 3. Covariate Balance — Standardised Mean Differences

In [ ]:
def compute_smd(df_sub, treatment_col, covariates):
    """Compute Standardised Mean Difference for each covariate."""
    smds = {}
    treat = df_sub[df_sub[treatment_col]==1]
    ctrl  = df_sub[df_sub[treatment_col]==0]
    for cov in covariates:
        m1, m0 = treat[cov].mean(), ctrl[cov].mean()
        s1, s0 = treat[cov].std(),  ctrl[cov].std()
        pooled_sd = np.sqrt((s1**2 + s0**2)/2)
        smd = (m1-m0)/pooled_sd if pooled_sd > 0 else 0
        smds[cov] = smd
    return pd.Series(smds)

smd_unadjusted = compute_smd(df, "statin", COVARIATES)

# Love plot
fig, ax = plt.subplots(figsize=(9,5))
y_pos = range(len(COVARIATES))
ax.barh(list(COVARIATES)[::-1], np.abs(smd_unadjusted.values[::-1]),
        color="#D65F5F", alpha=0.85, label="Unadjusted")
ax.axvline(0.10, color="black", ls="--", lw=1.5, label="SMD=0.10 threshold")
ax.axvline(0.25, color="gray",  ls=":", lw=1.5, label="SMD=0.25 (poor balance)")
ax.set_xlabel("Absolute SMD"); ax.set_title("Love Plot — Covariate Balance (Unadjusted)")
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/mod06/love_plot.png",bbox_inches="tight"); plt.show()
print("Unadjusted SMDs (higher = more imbalance):")
for cov, smd in smd_unadjusted.abs().sort_values(ascending=False).items():
    flag = " <- IMBALANCED" if smd > 0.10 else ""
    print(f"  {cov:20s}: {smd:.3f}{flag}")


## 4. Propensity Score Matching

In [ ]:
def nearest_neighbour_match(df, ps_col, treatment_col,
                            caliper=0.2, ratio=1, replace=False):
    """1:ratio nearest-neighbour PS matching with caliper."""
    treated = df[df[treatment_col]==1].copy()
    control = df[df[treatment_col]==0].copy()
    caliper_width = caliper * df[ps_col].std()

    matched_control_idx = []
    available = set(control.index)

    for _, row in treated.iterrows():
        ps_t = row[ps_col]
        candidates = [(abs(ps_t - control.loc[i,ps_col]), i)
                      for i in available if abs(ps_t - control.loc[i,ps_col]) <= caliper_width]
        candidates.sort()
        selected = [idx for _, idx in candidates[:ratio]]
        matched_control_idx.extend(selected)
        if not replace:
            available -= set(selected)

    matched_control = control.loc[matched_control_idx]
    matched_df = pd.concat([treated.iloc[:len(matched_control_idx)//ratio],
                            matched_control]).reset_index(drop=True)
    return matched_df

# Apply matching
df_matched = nearest_neighbour_match(df, "ps_gb", "statin", caliper=0.2, ratio=1)
print(f"Matched dataset: N={len(df_matched)} ({df_matched.statin.sum()} treated, {(1-df_matched.statin).sum()} control)")

smd_matched = compute_smd(df_matched, "statin", COVARIATES)

# Love plot comparing before/after matching
fig, ax = plt.subplots(figsize=(10,5))
n = len(COVARIATES)
y = np.arange(n)
ax.barh(y-0.2, np.abs(smd_unadjusted.values), 0.35, color="#D65F5F", alpha=0.8, label="Before matching")
ax.barh(y+0.2, np.abs(smd_matched.values), 0.35, color="#4878CF", alpha=0.8, label="After PS matching")
ax.set_yticks(y); ax.set_yticklabels(COVARIATES)
ax.axvline(0.10,color="black",ls="--",lw=1.5,label="SMD=0.10")
ax.set_xlabel("Absolute SMD"); ax.set_title("Love Plot: Before vs After PS Matching")
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/mod06/love_plot_matched.png",bbox_inches="tight"); plt.show()


## 5. ATT Estimation via Matching

In [ ]:
from sklearn.linear_model import LogisticRegression

def estimate_att_matching(df_matched, treatment, outcome):
    """Estimate ATT from matched dataset."""
    r1 = df_matched[df_matched[treatment]==1][outcome].mean()
    r0 = df_matched[df_matched[treatment]==0][outcome].mean()
    log_or = np.log(r1/(1-r1)) - np.log(r0/(1-r0))
    return {"RD":round(r1-r0,4),"RR":round(r1/r0,4),"log_OR":round(log_or,4),
            "OR":round(np.exp(log_or),4)}

att_match = estimate_att_matching(df_matched, "statin", "mi")
print("ATT (Matching):")
print(f"  Risk difference (RD): {att_match['RD']:.4f}")
print(f"  Risk ratio (RR)     : {att_match['RR']:.4f}")
print(f"  Odds ratio (OR)     : {att_match['OR']:.4f}")
print(f"  Log-odds            : {att_match['log_OR']:.4f}")
print(f"  True log-odds       : {TRUE_EFFECT:.4f}")
print(f"  Bias                : {att_match['log_OR']-TRUE_EFFECT:+.4f}")

# Bootstrap CI for ATT
def bootstrap_att(df_m, n_boot=500, seed=42):
    rng = np.random.default_rng(seed)
    ests = []
    for _ in range(n_boot):
        idx = rng.choice(len(df_m), len(df_m), replace=True)
        boot = df_m.iloc[idx]
        est = estimate_att_matching(boot,"statin","mi")
        ests.append(est["log_OR"])
    return np.percentile(ests,[2.5,97.5])

ci = bootstrap_att(df_matched)
print(f"  95% Bootstrap CI    : ({ci[0]:.4f}, {ci[1]:.4f})")


## 6. Inverse Probability of Treatment Weighting (IPTW)

In [ ]:
# ATE weights: 1/ps for treated, 1/(1-ps) for controls
# ATT weights: 1 for treated, ps/(1-ps) for controls

df["w_ate"] = np.where(df.statin==1, 1/df.ps_gb, 1/(1-df.ps_gb))
df["w_att"] = np.where(df.statin==1, 1.0, df.ps_gb/(1-df.ps_gb))

# Truncate extreme weights at 99th percentile
for w_col in ["w_ate","w_att"]:
    cap = np.percentile(df[w_col], 99)
    df[w_col] = df[w_col].clip(upper=cap)

print("Weight distributions:")
print(f"  ATE weights: mean={df.w_ate.mean():.2f}, max={df.w_ate.max():.2f}, ESS={1/((df.w_ate/df.w_ate.sum())**2).sum():.0f}")
print(f"  ATT weights: mean={df.w_att.mean():.2f}, max={df.w_att.max():.2f}")

# Weighted SMD (balance check for IPTW)
def weighted_smd(df, treatment, covariates, weights):
    smds = {}
    for cov in covariates:
        w1 = weights[df[treatment]==1]; x1 = df.loc[df[treatment]==1,cov]
        w0 = weights[df[treatment]==0]; x0 = df.loc[df[treatment]==0,cov]
        m1 = np.average(x1, weights=w1); m0 = np.average(x0, weights=w0)
        v1 = np.average((x1-m1)**2, weights=w1)
        v0 = np.average((x0-m0)**2, weights=w0)
        pooled = np.sqrt((v1+v0)/2)
        smds[cov] = (m1-m0)/pooled if pooled>0 else 0
    return pd.Series(smds)

smd_iptw = weighted_smd(df,"statin",COVARIATES,df.w_ate)

print("\nWeighted SMDs (IPTW ATE):")
for cov, smd in smd_iptw.abs().sort_values(ascending=False).items():
    flag = " <- still imbalanced!" if abs(smd)>0.10 else ""
    print(f"  {cov:20s}: {abs(smd):.3f}{flag}")


In [ ]:
# Estimate ATE using IPTW (Horvitz-Thompson estimator)
def iptw_estimate(df, treatment, outcome, weight_col):
    w = df[weight_col]
    r1 = (df[df[treatment]==1][outcome] * df.loc[df[treatment]==1,weight_col]).sum() / df.loc[df[treatment]==1,weight_col].sum()
    r0 = (df[df[treatment]==0][outcome] * df.loc[df[treatment]==0,weight_col]).sum() / df.loc[df[treatment]==0,weight_col].sum()
    log_or = np.log(r1/(1-r1)) - np.log(r0/(1-r0))
    return {"RD":round(r1-r0,4),"RR":round(r1/r0,4),
            "log_OR":round(log_or,4),"OR":round(np.exp(log_or),4)}

ate_iptw = iptw_estimate(df,"statin","mi","w_ate")
att_iptw = iptw_estimate(df,"statin","mi","w_att")

print("IPTW Results:")
print(f"{'':20s} {'ATE':>10s} {'ATT':>10s} {'True':>10s}")
print("-"*55)
for metric in ["log_OR","OR","RD"]:
    print(f"  {metric:18s} {ate_iptw[metric]:>10.4f} {att_iptw[metric]:>10.4f} "
          f"{'N/A':>10s}" if metric!="log_OR"
          else f"  {metric:18s} {ate_iptw[metric]:>10.4f} {att_iptw[metric]:>10.4f} {TRUE_EFFECT:>10.4f}")


## 7. Knowledge Check
1. Your PS model has AUC=0.85. Is this good or bad for causal inference?
2. After IPTW, SMD for LDL is 0.03. What does this mean for the analysis?
3. ATE and ATT differ by 0.08 log-odds. Under what circumstances would these be equal?
4. Extreme PS weights (e.g. 50x) can inflate variance. What are three remedies?
5. Implement doubly-robust estimation: combine outcome regression with IPTW weights.


In [ ]:
# Exercise 5 — Doubly Robust (AIPW) estimator
from sklearn.linear_model import LogisticRegression

# Outcome model (Q-model)
X_out = df[COVARIATES + ["statin"]]
lr_out = LogisticRegression(max_iter=500).fit(X_out, df["mi"])

# Predict potential outcomes
X1 = df[COVARIATES + ["statin"]].copy(); X1["statin"] = 1
X0 = df[COVARIATES + ["statin"]].copy(); X0["statin"] = 0
mu1 = lr_out.predict_proba(X1)[:,1]
mu0 = lr_out.predict_proba(X0)[:,1]

# AIPW correction terms
ps  = df.ps_gb.values
A   = df.statin.values
Y   = df.mi.values
# ATE AIPW
aipw_1 = mu1 + (A/ps) * (Y - mu1)
aipw_0 = mu0 + ((1-A)/(1-ps)) * (Y - mu0)
ate_aipw_rd = aipw_1.mean() - aipw_0.mean()
mu1_mean = mu1.mean(); mu0_mean = mu0.mean()
aipw_lor = np.log(mu1_mean/(1-mu1_mean)) - np.log(mu0_mean/(1-mu0_mean))

print("Doubly Robust (AIPW) Estimates:")
print(f"  ATE RD (AIPW)      : {ate_aipw_rd:.4f}")
print(f"  Approx log-OR      : {aipw_lor:.4f}")
print(f"  True log-OR        : {TRUE_EFFECT:.4f}")
print(f"  Bias               : {aipw_lor-TRUE_EFFECT:+.4f}")
print()
print("AIPW is consistent if EITHER the PS model OR the outcome model is correct.")


---
**Next -> NB-03: Difference-in-Differences**